In [3]:
from google import genai
from dotenv import load_dotenv

import json
import cv2
import numpy as np

load_dotenv()

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain how AI works in a few words",
)

print(response.text)



AI learns from data to recognize patterns and make decisions.


In [4]:
# === Config ===
# scorecard_file_path = "/workspaces/ARC-AGI-3-Agents/recordings/ls20-016295f7601e.random.80.723aa0c3-e6d8-4693-9ae3-c110dd06e58a.recording.jsonl"
scorecard_file_path = "recordings/ls20-016295f7601e.random.300.3eb1cdcc-73ea-4f60-b75b-d57c411c14c1.recording.jsonl"
video_output_path = "output_video.mp4"
pixel_size = 10
fps = 10

# === Color palette (from key_colors as hex) ===
key_colors = {
    0: "#FFFFFF", 1: "#CCCCCC", 2: "#999999", 3: "#666666",
    4: "#333333", 5: "#000000", 6: "#E53AA3", 7: "#FF7BCC",
    8: "#F93C31", 9: "#1E93FF", 10: "#88D8F1", 11: "#FFDC00",
    12: "#FF851B", 13: "#921231", 14: "#4FCC30", 15: "#A356D6"
}
# Convert to BGR for OpenCV
palette = np.array([tuple(int(color[i:i+2], 16) for i in (1, 3, 5))[::-1] for color in key_colors.values()], dtype=np.uint8)

# === Load JSONL ===
with open(scorecard_file_path, "r") as file:
    grid_jsons = [json.loads(line) for line in file]

# === Find frame size from first frame ===
first_grid = grid_jsons[0]["data"]["frame"][0]
h, w = len(first_grid), len(first_grid[0])
frame_size = (w * pixel_size, h * pixel_size)

# === Setup video writer ===
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video = cv2.VideoWriter(video_output_path, fourcc, fps, frame_size)

# === Frame processing loop ===
for grid_json in grid_jsons[:-1]:
    for grid in grid_json["data"]["frame"]:
        grid_array = np.array(grid, dtype=np.uint8)
        color_image = palette[grid_array]  # shape: (H, W, 3)
        scaled_image = cv2.resize(color_image, frame_size, interpolation=cv2.INTER_NEAREST)
        video.write(scaled_image)

video.release()
print(f"✅ Video saved to {video_output_path}")


✅ Video saved to output_video.mp4


In [12]:

from google.genai import types

INITIAL_GAME_ANALYSIS_PROMPT = """This video is a random actions (WASD and click) moves taken on unkown game.

The game is designed based on below Constraints
- Easy for humans (can pick it up in <1 min of game play)
- Core Knowledge Priors (no language, trivia, cultural symbols)
- Should require no instructions to play
- Should be fun for humans and playable in 5-10 minutes
- Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)

Here are the actions that your player can take
W: Move Up
A: Move Left
S: Move Down
D: Move Right
(x,y): Click on the area by giving x,y space (x: <0, 63>, y: <0, 63>)
Sometimes, some actions has no effect. 

Give 10 hypothesis with hypothesis test action sequence to explore the game and understand its mechanics, objectives, and challenges. 
This action sequence will be played by human player to test the hypothesis.
"""

# Only for videos of size <20Mb
video_file_name = video_output_path
video_bytes = open(video_file_name, 'rb').read()

random_explorer_agent_response = client.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=types.Content(
        parts=[
            types.Part(
                inline_data=types.Blob(data=video_bytes, mime_type='video/mp4')
            ),
            types.Part(text=INITIAL_GAME_ANALYSIS_PROMPT)
        ]
    )
)

In [13]:
print(random_explorer_agent_response.text)

Here are 10 hypotheses about the game's mechanics, objectives, and challenges, each with a proposed test action sequence for a human player. For movement, assume the player navigates with WASD to reach the described position. For interaction, `(X,Y)` indicates a mouse click at those pixel coordinates on the screen.

---

**1. Hypothesis: The primary objective is to push the orange/blue block into the black goal square.**
*   **Test Action Sequence:**
    1.  **Move player (blue square) to the left of the orange/blue block.** (Use WASD to position your player around (31, 32)).
    2.  **(32, 32)** (Click on the orange/blue block to push it right, observing the white T-shape appear.)
    3.  **Strategically maneuver the player (using WASD) around obstacles and the white T-shapes, then continue pushing the orange/blue block (by moving adjacent and clicking its coordinates) until it enters the black goal area at (52, 17).** (This will be a sequence of many moves and clicks, aiming to repli

In [ ]:
from enum import Enum
from google.genai import types

HYPOTHESIS_SELECTOR_PROMPT = """Among these hypothesis, which hypothesis focuses on win

This is a list of hypothesis:
{hypothesis_list}
"""

HYPOTHESIS_ACTION_NAVIGATOR_PROMPT = """You are an agent playing a dynamic game. Your objective is to
achieve the below hypothesis by taking actions based on the game grid.

<hypothesis>
{hypothesis}
</hypothesis>

One action produces one Frame. One Frame is made of one or more sequential
Grids. Each Grid is a matrix size INT<0,63> by INT<0,63> filled with
INT<0,15> values.

AVAILABLE ACTIONS:
- Move Up (W)
- Move Left (A)
- Move Down (S)
- Move Right (D)
- Click on the area by giving x,y space CLICK(x,y)
- hypothesis got achieved (ACHIEVED)
- Wrong hypothesis (WRONG_HYPOTHESIS)

Call exactly one action

Respond with a JSON:
{{
  "reason": "Your reason for the action (max 50 words)",
  "action": "The action you want to take"
}}
"""

class HypothesisAction(Enum):
    MOVE_UP = "W"
    MOVE_LEFT = "A"
    MOVE_DOWN = "S"
    MOVE_RIGHT = "D"
    CLICK = "CLICK"
    ACHIEVED = "ACHIEVED"
    WRONG_HYPOTHESIS = "WRONG_HYPOTHESIS"

def select_action_for_hypothesis(hypothesis: str) -> dict:
    response = client.models.generate_content(
        model='models/gemini-2.5-flash',
        contents=types.Content(
            parts=[types.Part(text=HYPOTHESIS_ACTION_NAVIGATOR_PROMPT.format(hypothesis=hypothesis))]
        )
    )
    
    # Parse JSON response
    try:
        response_text = response.text
        response_text = response_text.removeprefix("```json").removesuffix("```").strip()
        response_text = response_text.strip()
        parsed = json.loads(response_text)
        reason = parsed.get("reason", "No reason provided.")
        action_text = parsed["action"].strip()

        # Determine the corresponding action
        if action_text == "W":
            action = HypothesisAction.MOVE_UP
        elif action_text == "A":
            action = HypothesisAction.MOVE_LEFT
        elif action_text == "S":
            action = HypothesisAction.MOVE_DOWN
        elif action_text == "D":
            action = HypothesisAction.MOVE_RIGHT
        elif action_text.startswith("CLICK"):
            coords = action_text[6:-1].split(',')
            action = (HypothesisAction.CLICK, (int(coords[0]), int(coords[1])))
        elif action_text == "ACHIEVED":
            action = HypothesisAction.ACHIEVED
        elif action_text == "WRONG_HYPOTHESIS":
            action = HypothesisAction.WRONG_HYPOTHESIS
        else:
            raise ValueError(f"Unknown action text: {action_text}")

        return {
            "reason": reason,
            "action": action
        }

    except (json.JSONDecodeError, KeyError, ValueError) as e:
        raise ValueError(f"Failed to parse model response: {response.text}") from e

In [17]:
hypothesis = """Hypothesis 1: The primary objective is to push the orange/blue block into the black goal square.
Because Hypothesis 1 is the only one that describes exactly the action that produces a *persistent progress change*—the purple “lit” square on the progress bar—without triggering a reset. In every other hypothesis you’re either testing death/reset conditions (blocks going off‑screen or into walls), temporary obstacles (the white T’s), respawns, or UI counters (lives and attempts).

Hypothesis 1 alone matches what we intuitively think of as the “win” event in the footage:

* **Successful goal‑entry**: The orange/blue block ends up fully inside the black square.
* **Positive feedback**: A purple square lights up on the top bar—unlike failures, there’s no red flash or reset.
* **No reset**: The level continues from that new state, proving it’s not just a mechanic or death test but *progress toward victory.*

That combination of “block in goal → purple progress marker → no reset” is precisely the signature of a completed objective, i.e. the win condition.
"""
select_action_for_hypothesis(hypothesis)

{'reason': 'Hypothesis 1, outlining the primary objective of pushing the orange/blue block into the black goal, has been successfully identified and understood as the win condition. This confirms the objective for the agent before game state is provided.',
 'action': <HypothesisAction.ACHIEVED: 'ACHIEVED'>}

In [7]:
EXPLORER_AGENT_PROMPT = """
You are an intelligent agent in a novel, minimal-instruction game. Your objective is to **explore** the environment to:

- Understand any **unsure or ambiguous UI elements**
- Identify the **rules or hidden mechanics** of the game
- Discover what leads to **winning or progressing**

### Guidelines for Exploration:
- If an element looks uncertain or unexplained, **approach or interact** with it to gain clarity.
- If you cannot move toward an element, attempt to **click on its coordinates** using (x, y).
- Coordinate system:  
  x: 0 to 63  
  y: 0 to 63

### Game Design Principles (Meta Context):
- Designed for humans to understand within 1 minute
- No language, trivia, or culture-specific knowledge required
- Should require zero external instructions
- Meant to be fun, intuitive, and completable in 5–10 minutes
- May include mechanics like hidden state, theory of mind, long-term planning, or navigation involving other agents

### Your Task:
Analyze your current state. Focus on elements you do not fully understand or that might relate to:
- Scoring
- Goal conditions
- Interactions (objects, agents, mechanics)
- Hidden patterns or state changes

Current state
{random_analysis}

Use **W, A, S, D** to move or click at specific coordinates if movement isn't possible.

### Output:
Respond in JSON with your planned action and reasoning behind it:

{{
    "reason": "Explain which UI element or mechanic you are trying to understand or clarify, and why this particular action (move or click) will help you explore or uncover the game's rules or win condition.",
    "action": "W, A, S, D, or (x,y)"
}}
"""


prompt = EXPLORER_AGENT_PROMPT.format(
    random_analysis="""```json
{
  "ui_elements": [
    {
      "element_id": "game_board",
      "type": "play_area",
      "description": "The main grid-based game board where the player manipulates objects.",
      "confidence": "sure",
      "thoughts": "This is clearly the interactive space for the game mechanics."
    },
    {
      "element_id": "player_character",
      "type": "player_avatar",
      "description": "The controllable L-shaped white piece with a blue pixel attached. The user moves this object using WASD.",
      "confidence": "sure",
      "thoughts": "Its movement directly correlates with user input shown by the video's actions."
    },
    {
      "element_id": "goal_object_main",
      "type": "interactive_object",
      "description": "The blue rectangular block with an orange top. This is the primary object that needs to be moved or interacted with to progress, seemingly pushed by the player.",
      "confidence": "sure",
      "thoughts": "The player consistently pushes this object towards the target zone."
    },
    {
      "element_id": "goal_object_helper",
      "type": "interactive_object",
      "description": "The small L-shaped white piece with a blue pixel, similar to the player character but smaller. It also moves and appears to be another object the player can push or that interacts with the main goal object.",
      "confidence": "sure",
      "thoughts": "It moves independently of the player but can be pushed, and its interaction with the main goal object seems crucial."
    },
    {
      "element_id": "target_zone",
      "type": "goal_area",
      "description": "A black square with a blue pixel and a small white L-shape inside (top-right of the play area). This appears to be the destination or 'goal' for the main blue/orange block.",
      "confidence": "sure",
      "thoughts": "When the blue/orange block reaches this area, a visual change (purple squares light up) occurs, indicating success."
    },
    {
      "element_id": "puzzle_progress_indicator",
      "type": "progress_bar",
      "description": "A series of small grey squares at the top-left of the screen. They turn purple one by one when a sub-goal or individual puzzle stage is completed.",
      "confidence": "sure",
      "thoughts": "Each time the blue/orange block successfully enters the target zone (or triggers the next step), one grey square turns purple, indicating progression through a sequence of puzzles or steps within a larger level."
    },
    {
      "element_id": "overall_game_progress_markers",
      "type": "game_state_indicator",
      "description": "Three red squares at the top-right of the screen. When all 'puzzle progress indicator' squares turn purple, one of these red squares turns grey, and the 'puzzle progress indicator' resets. They likely represent major milestones or completed 'sets' of puzzles within the overall game.",
      "confidence": "not sure",
      "thoughts": "Their function isn't entirely clear. They don't appear to be 'lives' as they change upon success, not failure. They seem to mark completion of a larger 'challenge' or 'world' once a full set of purple squares is achieved. They could be 'major levels completed' or 'attempts remaining' for a meta-goal, but the latter seems less likely given they turn grey on success."
    },
    {
      "element_id": "bottom_static_bar",
      "type": "unknown_indicator",
      "description": "A series of grey squares at the bottom of the screen. Their function is not evident from the video as they remain static and do not change. Potentially an inactive inventory, a move counter that wasn't activated, or another unrevealed game state display.",
      "confidence": "not sure",
      "thoughts": "They consistently remain grey throughout the video. Without further interaction or context, their purpose is unknown."
    }
  ]
}
```"""
)
print(prompt)


You are an intelligent agent in a novel, minimal-instruction game. Your objective is to **explore** the environment to:

- Understand any **unsure or ambiguous UI elements**
- Identify the **rules or hidden mechanics** of the game
- Discover what leads to **winning or progressing**

### Guidelines for Exploration:
- If an element looks uncertain or unexplained, **approach or interact** with it to gain clarity.
- If you cannot move toward an element, attempt to **click on its coordinates** using (x, y).
- Coordinate system:  
  x: 0 to 63  
  y: 0 to 63

### Game Design Principles (Meta Context):
- Designed for humans to understand within 1 minute
- No language, trivia, or culture-specific knowledge required
- Should require zero external instructions
- Meant to be fun, intuitive, and completable in 5–10 minutes
- May include mechanics like hidden state, theory of mind, long-term planning, or navigation involving other agents

### Your Task:
Analyze your current state. Focus on element

In [29]:
import json
from pydantic import BaseModel, Field
class ExplorerAction(BaseModel):
    reason: str = Field(..., description="Reason for the action")
    action: str = Field(..., description="Action to take (W, A, S, D, or (x,y))")

explorer_agent_action_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=EXPLORER_AGENT_PROMPT.format(
        random_analysis=random_explorer_agent_response.text
    ),
)


# Parse the response into the ExplorerAction model
explorer_agent_action_response = explorer_agent_action_response.text.removeprefix("```json\n").removesuffix("\n```")

explorer_agent_action = ExplorerAction.model_validate_json(explorer_agent_action_response)
explorer_agent_action

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC remote call 1 is done.


ExplorerAction(reason="The primary goal is to explore 'unsure' elements to understand game rules and win conditions. The 'overall_game_progress_markers' are described as changing upon successful completion of 'sets' of puzzles. The most direct way to explore this element is to continue playing the game, solve the current puzzle, and observe its reaction to further progression. This requires moving the 'player_character' on the 'game_board' using WASD. Additionally, the 'bottom_static_bar' is an 'unknown_indicator' whose function is not evident. By making a move, we can observe if this bar acts as a move counter, or if its state changes in response to player actions or puzzle progression, helping to clarify its purpose. Since movement is possible and directly contributes to understanding the more critical 'overall_game_progress_markers', a WASD input is the appropriate action to explore both unsure elements.", action='W')

In [26]:
from typing import List
from PIL import Image, ImageDraw, ImageFont
import io
import logging
# You can define this globally or inside a class/module
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)  # Or DEBUG for more detail

def generate_grid_image_with_zone(
        grid: List[List[int]], cell_size: int = 40, zone_size: int = 20
    ) -> bytes:
        """Generate PIL image of the grid with colored cells and zone coordinates."""
        if not grid or not grid[0]:
            # Create empty image
            img = Image.new("RGB", (200, 200), color="black")
            buffer = io.BytesIO()
            img.save(buffer, format="PNG")
            return buffer.getvalue()

        height = len(grid)
        width = len(grid[0])

        # Create image
        img = Image.new("RGB", (width * cell_size, height * cell_size), color="white")
        draw = ImageDraw.Draw(img)

        # Color mapping for grid cells
        key_colors = {
            0: "#FFFFFF",
            1: "#CCCCCC",
            2: "#999999",
            3: "#666666",
            4: "#333333",
            5: "#000000",
            6: "#E53AA3",
            7: "#FF7BCC",
            8: "#F93C31",
            9: "#1E93FF",
            10: "#88D8F1",
            11: "#FFDC00",
            12: "#FF851B",
            13: "#921231",
            14: "#4FCC30",
            15: "#A356D6"
        }

        # Draw grid cells
        for y in range(height):
            for x in range(width):
                color = key_colors.get(grid[y][x], "#888888")  # default: floor

                # Draw cell
                draw.rectangle(
                    [
                        x * cell_size,
                        y * cell_size,
                        (x + 1) * cell_size,
                        (y + 1) * cell_size,
                    ],
                    fill=color,
                    outline="#000000",
                    width=1,
                )

        # Draw zone coordinates and borders
        for y in range(0, height, zone_size):
            for x in range(0, width, zone_size):
                # Draw zone coordinate label
                try:
                    font = ImageFont.load_default()
                    zone_text = f"({x},{y})"
                    draw.text(
                        (x * cell_size + 2, y * cell_size + 2),
                        zone_text,
                        fill="#FFFFFF",
                        font=font,
                    )
                except (ImportError, OSError) as e:
                    logger.debug(f"Could not load font for zone labels: {e}")
                except Exception as e:
                    logger.error(f"Failed to draw zone label at ({x},{y}): {e}")

                # Draw zone boundary
                zone_width = min(zone_size, width - x) * cell_size
                zone_height = min(zone_size, height - y) * cell_size
                draw.rectangle(
                    [
                        x * cell_size,
                        y * cell_size,
                        x * cell_size + zone_width,
                        y * cell_size + zone_height,
                    ],
                    fill=None,
                    outline="#FFD700",  # gold border for zone
                    width=2,
                )

        # Convert to bytes
        buffer = io.BytesIO()
        img.save(buffer, format="PNG")
        return buffer.getvalue()

grid_image = generate_grid_image_with_zone(grid=grid_jsons[0]["data"]["frame"][0])

In [28]:
OBSERVATION_AGENT_PROMPT = """You are a coach of a unknown game

You have an sure and unsure observation of UI elements and other elements in the game.
{observation}
Your player has taken some move to explore and win the game. 

This is a image of the game after the move is made by your player.

The game is designed based on below Constraints
- Easy for humans (can pick it up in <1 min of game play)
- Core Knowledge Priors (no language, trivia, cultural symbols)
- Should require no instructions to play
- Should be fun for humans and playable in 5-10 minutes
- Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)

here are the actions that your player can take
W: Move Up
A: Move Left
S: Move Down
D: Move Right
(x,y): Click on the area by giving x,y space (x: <0, 63>, y: <0, 63>)

Now, you need to change the observation based on the moves made by your player.
the player has taken the action {action}.
the reason for the action is {reason}.

Give json output with the updated observation and the action taken by your player
"""

# Create the prompt with text and multiple images
observation_agent_response = client.models.generate_content(

    model="gemini-2.5-flash",
    contents=[
        OBSERVATION_AGENT_PROMPT.format(
        observation=random_explorer_agent_response.text,
        action=explorer_agent_action.action,
        reason=explorer_agent_action.reason
    ),
        types.Part.from_bytes(
            data=grid_image,
            mime_type='image/png'
        )
    ]
)

print(observation_agent_response.text)

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC remote call 1 is done.


```json
{
  "updated_observation": {
    "ui_elements": [
      {
        "element_id": "game_board",
        "type": "play_area",
        "description": "The main grid-based game board where the player manipulates objects.",
        "confidence": "sure",
        "thoughts": "This is clearly the interactive space for the game mechanics."
      },
      {
        "element_id": "player_character",
        "type": "player_avatar",
        "description": "The controllable L-shaped white piece with a blue pixel attached. The user moves this object using WASD.",
        "confidence": "sure",
        "thoughts": "Its movement directly correlates with user input shown by the video's actions."
      },
      {
        "element_id": "goal_object_main",
        "type": "interactive_object",
        "description": "The blue rectangular block with an orange top. This is the primary object that needs to be moved or interacted with to progress, seemingly pushed by the player.",
        "confidence": 